   # 03 · Bronze: meldingen
   Loads the raw CSV files from the landing volume into `eindhoven.bronze.meldingen` with Auto Loader.
   - incremental (checkpoint): already-processed files are skipped
   - metadata: `bronbestand`, `ingelezen_op`

In [0]:
from pyspark.sql import functions as F
df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("sep", ";")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", "/Volumes/eindhoven/landing/aanlevering/_schema/meldingen")
        .option("pathGlobFilter", "*.csv")
        .load("/Volumes/eindhoven/landing/aanlevering/meldingen")
        .withColumn("bronbestand", F.col("_metadata.file_path"))
        .withColumn("ingelezen_op", F.current_timestamp())

    .writeStream
        .option("checkpointLocation", "/Volumes/eindhoven/landing/aanlevering/_checkpoint/meldingen")
        .trigger(availableNow=True)
        .toTable("eindhoven.bronze.meldingen")
        .awaitTermination()
)